# Chess ELO Prediction with Neural Networks

**Goal:** Given a completed chess game, predict both players' Elo ratings
from their playing style — not just the result.

The intuition is that *how* you play reveals your strength: strong players
manage the clock more carefully, avoid blunders, and control territory more
consistently than weaker players.

## Dataset
**Lichess open database, May 2017** — rated standard games.
Each game is a PGN string with move annotations and optional clock times.

## Four architectures compared on the same data

| Model | Sequence processing |
|---|---|
| **BiLSTM** | Two-layer bidirectional LSTM with soft attention |
| **Residual MLP** | No sequences — pure engineered features |
| **1D CNN** | Three parallel conv branches (kernels 3, 5, 7) |
| **Transformer** | Multi-head self-attention encoder |

Each model predicts a **probability distribution** over 40 Elo bins (0–4000),
trained with **KL divergence** against a Gaussian soft target.


In [ ]:
import io, pickle, re, time, warnings
from copy import deepcopy
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import zstandard as zstd
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import chess_features_final as cff

warnings.filterwarnings('ignore')
plt.style.use('seaborn-v0_8-whitegrid')
print(f'PyTorch {torch.__version__}  |  CUDA: {torch.cuda.is_available()}')


## Configuration

All experiment settings live here. Change `RUN_*` flags at the top to choose which models to train.


In [ ]:
# ── Paths ─────────────────────────────────────────────────────────────────────
DATA_PATH = '../../data/elias_data/lichess_db_standard_rated_2017-05.pgn.zst'
CACHE_DIR = Path('cache');       CACHE_DIR.mkdir(exist_ok=True)
CKPT_DIR  = Path('checkpoints'); CKPT_DIR.mkdir(exist_ok=True)

# ── Dataset ────────────────────────────────────────────────────────────────────
MAX_GAMES   = 1_000_000
TIME_CONTROL = None   # e.g. "600+0" — exact match on TimeControl header (None = any)
TC_BASE_MIN  = 600   # minimum base time in seconds  (None = no bound)
TC_BASE_MAX  = 600   # maximum base time in seconds  (None = no bound)
TC_INC_MIN   = None   # minimum increment in seconds  (None = no bound)
TC_INC_MAX   = None   # maximum increment in seconds  (None = no bound)
SEED        = 42

# ── Which models to run (True / False) ────────────────────────────────────────
RUN_BILSTM      = True
RUN_RESIDUAL    = True
RUN_CNN         = True
RUN_TRANSFORMER = True

# ── Shared training settings ───────────────────────────────────────────────────
BATCH_SIZE   = 128
NUM_EPOCHS   = 10
LR           = 3e-4
WEIGHT_DECAY = 1e-4
DROPOUT      = 0.30
PATIENCE     = 3

# ── Elo bins ───────────────────────────────────────────────────────────────────
N_BINS    = 40      # 0 to 4000 in 100-pt steps
ELO_SIGMA = 200     # Gaussian spread for soft targets

# ── Sequence settings (BiLSTM, CNN, Transformer) ───────────────────────────────
MIN_PLIES   = 12
MAX_SEQ_LEN = 120
EMBED_DIM   = 64

# ── Model-specific dims ────────────────────────────────────────────────────────
BILSTM_HIDDEN   = 128
RESIDUAL_DIM    = 256
RESIDUAL_BLOCKS = 4
CNN_FILTERS     = 64
TRANS_DIM       = 128
TRANS_HEADS     = 4
TRANS_LAYERS    = 2
TRANS_FF_DIM    = 256

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BIN_EDGES   = np.linspace(0, 4000, N_BINS + 1)
BIN_CENTERS = ((BIN_EDGES[:-1] + BIN_EDGES[1:]) / 2).astype(np.float32)

# ── HPO settings ─────────────────────────────────────────────────────────────
# Set HPO_* = True to tune that model before full training (uses Optuna).
# Results are cached, so re-running is instant unless you delete the cache.
HPO_BILSTM      = False
HPO_RESIDUAL    = False
HPO_CNN         = False
HPO_TRANSFORMER = False

HPO_TRIALS = 15       # Optuna trials per model
HPO_GAMES  = 30_000   # games to train on per trial (keeps trials fast)
HPO_EPOCHS = 3        # training epochs per trial

print(f'Device : {device}')
print(f'Models : BiLSTM={RUN_BILSTM}  Residual={RUN_RESIDUAL}  CNN={RUN_CNN}  Transformer={RUN_TRANSFORMER}')
print(f'HPO    : BiLSTM={HPO_BILSTM}  Residual={HPO_RESIDUAL}  CNN={HPO_CNN}  Transformer={HPO_TRANSFORMER}')


## 1 · Data Loading

Lichess publishes monthly PGN dumps compressed with zstandard (`.zst`).
We stream-parse the file game-by-game so we never decompress the entire file at once.

Each game becomes one row: header fields (White, Black, WhiteElo, BlackElo,
TimeControl, …) plus a `Moves` column with the full annotated PGN move string.


In [ ]:
def load_pgn_zst(path, max_games):
    # Stream-parse a zstd-compressed PGN file into a DataFrame
    records = []
    with open(path, 'rb') as fh:
        dctx   = zstd.ZstdDecompressor()
        stream = io.TextIOWrapper(dctx.stream_reader(fh), encoding='utf-8', errors='replace')
        headers, moves_lines, in_moves = {}, [], False
        for line in stream:
            line = line.strip()
            if line.startswith('['):
                in_moves = False
                m = re.match(r'\[(\w+)\s+\"(.*)\"\]', line)
                if m: headers[m.group(1)] = m.group(2)
            elif line == '':
                if headers and moves_lines:
                    headers['Moves'] = ' '.join(moves_lines)
                    records.append(headers)
                    headers, moves_lines = {}, []
                    if len(records) >= max_games: break
                elif headers:
                    in_moves = True
            else:
                if in_moves or headers: moves_lines.append(line)
        if headers and moves_lines and len(records) < max_games:
            headers['Moves'] = ' '.join(moves_lines)
            records.append(headers)
    return pd.DataFrame(records)


t0     = time.time()
df_raw = load_pgn_zst(DATA_PATH, MAX_GAMES)
print(f'Loaded {len(df_raw):,} games  ({time.time()-t0:.1f}s)')

for col in ['WhiteElo', 'BlackElo']:
    df_raw[col] = pd.to_numeric(df_raw[col], errors='coerce')
df_raw.dropna(subset=['WhiteElo', 'BlackElo', 'Moves'], inplace=True)
df_raw[['WhiteElo', 'BlackElo']] = df_raw[['WhiteElo', 'BlackElo']].astype(int)

TC_RE = re.compile(r'^(\d+)\+(\d+)$')
def parse_tc(s):
    m = TC_RE.match(str(s))
    return (int(m.group(1)), int(m.group(2))) if m else (np.nan, np.nan)

df_raw[['tc_base', 'tc_inc']] = df_raw['TimeControl'].apply(lambda x: pd.Series(parse_tc(x)))
df_raw.dropna(subset=['tc_base'], inplace=True)
df_raw['tc_base'] = df_raw['tc_base'].astype(int)
df_raw['tc_inc']  = df_raw['tc_inc'].astype(int)

if TIME_CONTROL is not None:
    df_raw = df_raw[df_raw['TimeControl'] == TIME_CONTROL]
if TC_BASE_MIN is not None:
    df_raw = df_raw[df_raw['tc_base'] >= TC_BASE_MIN]
if TC_BASE_MAX is not None:
    df_raw = df_raw[df_raw['tc_base'] <= TC_BASE_MAX]
if TC_INC_MIN is not None:
    df_raw = df_raw[df_raw['tc_inc'] >= TC_INC_MIN]
if TC_INC_MAX is not None:
    df_raw = df_raw[df_raw['tc_inc'] <= TC_INC_MAX]
df_raw = df_raw.reset_index(drop=True)

if 'Termination' in df_raw.columns:
    bad = {'Time forfeit', 'Abandoned', 'Unterminated'}
    df_raw = df_raw[~df_raw['Termination'].isin(bad)].reset_index(drop=True)

print(f'After filters : {len(df_raw):,} games')
print(f'Elo range     : {df_raw.WhiteElo.min()}–{df_raw.WhiteElo.max()}')

fig, axes = plt.subplots(1, 2, figsize=(11, 3))
for ax, col, c in zip(axes, ['WhiteElo', 'BlackElo'], ['steelblue', 'coral']):
    ax.hist(df_raw[col], bins=50, color=c, edgecolor='none', alpha=0.85)
    ax.axvline(df_raw[col].median(), color='k', lw=1.5, ls='--',
               label=f'Median {df_raw[col].median():.0f}')
    ax.set_xlabel('Elo'); ax.set_ylabel('Games'); ax.set_title(col); ax.legend()
plt.suptitle('Elo Distribution', fontsize=12)
plt.tight_layout(); plt.show()


## 2 · Feature Engineering

We extract two groups of features that summarise a player's style without
requiring real-time sequence processing at training time:

**Board features** (replayed via `chess_features_final.py`, ~30 columns):
captures (density, first move, pawn vs piece), checks given, castling side/move,
piece activity (consecutive same piece, queen before move 10), territory depth,
and engine quality metrics (avg centipawn loss, blunders — NaN → 0 when evals absent).

**Clock features** (~10 columns):
Average, std, max time spent per move; time pressure in the last 5 moves;
opening pace in the first 5 moves. All normalised by the base time-control.

Both are cached to disk after the first run.


In [ ]:
# ── Board features ────────────────────────────────────────────────────────────
board_cache = CACHE_DIR / f'board_{MAX_GAMES}.pkl'
if board_cache.exists():
    df_board = pd.read_pickle(board_cache)
    print(f'Board features loaded from cache  ({df_board.shape[1]} cols)')
else:
    print('Extracting board features (a few minutes)...')
    t0 = time.time()
    df_board = cff.extract_features_dataframe(df_raw)
    df_board.to_pickle(board_cache)
    print(f'Done  ({df_board.shape[1]} cols  {time.time()-t0:.0f}s) — cached')

# ── Clock features ─────────────────────────────────────────────────────────────
clock_cache = CACHE_DIR / f'clock_{MAX_GAMES}.pkl'
CLK_RE = re.compile(r'\[%clk\s+(\d+):(\d+):(\d+(?:\.\d+)?)\]')

def extract_clocks(pgn):
    return [int(h)*3600 + int(m)*60 + float(s) for h, m, s in CLK_RE.findall(pgn)]

def clock_features(pgn, tc_base):
    clocks = extract_clocks(pgn)
    empty  = {k: np.nan for k in [
        'w_avg_time_norm','w_std_time_norm','w_max_time_norm','w_time_pressure','w_opening_pace',
        'b_avg_time_norm','b_std_time_norm','b_max_time_norm','b_time_pressure','b_opening_pace',
    ]}
    if len(clocks) < 4 or tc_base <= 0:
        return empty
    def stats(clk):
        spent = [max(0., clk[i] - clk[i+1]) for i in range(len(clk)-1)]
        if not spent:
            return dict(avg=np.nan, std=0., mx=np.nan, pressure=np.nan, opening=np.nan)
        s = np.array(spent)
        return dict(
            avg      = float(s.mean()                    / tc_base),
            std      = float(s.std()                     / tc_base) if len(s) > 1 else 0.,
            mx       = float(s.max()                     / tc_base),
            pressure = float(s[-min(5, len(s)):].mean()  / tc_base),
            opening  = float(s[:min(5, len(s))].mean()   / tc_base),
        )
    ws, bs = stats(clocks[0::2]), stats(clocks[1::2])
    return {
        'w_avg_time_norm': ws['avg'], 'w_std_time_norm': ws['std'],
        'w_max_time_norm': ws['mx'],  'w_time_pressure': ws['pressure'], 'w_opening_pace': ws['opening'],
        'b_avg_time_norm': bs['avg'], 'b_std_time_norm': bs['std'],
        'b_max_time_norm': bs['mx'],  'b_time_pressure': bs['pressure'], 'b_opening_pace': bs['opening'],
    }

if clock_cache.exists():
    df_clock = pd.read_pickle(clock_cache)
    print(f'Clock features loaded from cache  ({df_clock.shape[1]} cols)')
else:
    print('Extracting clock features...')
    t0 = time.time()
    df_clock = pd.DataFrame(
        [clock_features(row.Moves, row.tc_base) for _, row in df_raw.iterrows()],
        index=df_raw.index
    )
    df_clock.to_pickle(clock_cache)
    print(f'Done  ({df_clock.shape[1]} cols  {time.time()-t0:.0f}s) — cached')


In [ ]:
# ── Assemble full DataFrame ─────────────────────────────────────────────────
df = pd.concat([df_raw, df_board.reindex(df_raw.index), df_clock.reindex(df_raw.index)], axis=1)

skip = {'WhiteElo', 'BlackElo', 'Moves', 'White', 'Black', 'Event', 'Site',
        'Date', 'Round', 'Result', 'TimeControl', 'Termination', 'UTCDate', 'UTCTime'}

static_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in skip]
n_static    = len(static_cols)
print(f'Static features: {n_static}')
print(sorted(static_cols))


## 3 · Preprocessing

### Soft Elo targets
Rather than predicting a single number, we predict a **probability distribution**
over 40 Elo bins. The target for a player with true rating *e* is a Gaussian with
σ = 200 centred on *e*, normalised to sum to 1.

Training with **KL divergence** against this soft target naturally captures
rating uncertainty — the model learns that 1400 and 1600 are close, not unrelated.

### Weighted sampling
Very low and very high Elo games are rare. **Inverse-frequency weighted sampling**
oversamples these games during training so the model generalises across all skill levels.


In [ ]:
def make_elo_dist(elo):
    # Gaussian soft label centred on elo with std=ELO_SIGMA
    p = np.exp(-0.5 * ((BIN_CENTERS - elo) / ELO_SIGMA) ** 2)
    return (p / p.sum()).astype(np.float32)

# Sanity check — plot soft targets for three rating levels
fig, ax = plt.subplots(figsize=(8, 3))
for elo, color in [(900, 'steelblue'), (1600, 'forestgreen'), (2300, 'crimson')]:
    ax.plot(BIN_CENTERS, make_elo_dist(elo), label=f'Elo {elo}', color=color, lw=2)
ax.set_xlabel('Bin centre (Elo)'); ax.set_ylabel('Probability')
ax.set_title(f'Soft Gaussian targets  (sigma = {ELO_SIGMA})')
ax.legend(); plt.tight_layout(); plt.show()


In [ ]:
# ── Train / val / test split (80 / 10 / 10) ────────────────────────────────
avg_elo = (df['WhiteElo'] + df['BlackElo']) / 2
strata  = pd.cut(avg_elo, bins=10, labels=False).fillna(0).astype(int)

idx_all = df.index.tolist()
idx_tr, idx_tmp = train_test_split(idx_all, test_size=0.20, stratify=strata, random_state=SEED)
idx_val, idx_te = train_test_split(idx_tmp, test_size=0.50,
                                   stratify=strata.loc[idx_tmp], random_state=SEED)
print(f'Train {len(idx_tr):,}  val {len(idx_val):,}  test {len(idx_te):,}')

# ── Scale static features (fit on train only) ────────────────────────────────
df[static_cols] = df[static_cols].astype(float)
scaler = StandardScaler()
df.loc[idx_tr,  static_cols] = scaler.fit_transform(df.loc[idx_tr,  static_cols].fillna(0))
df.loc[idx_val, static_cols] = scaler.transform    (df.loc[idx_val, static_cols].fillna(0))
df.loc[idx_te,  static_cols] = scaler.transform    (df.loc[idx_te,  static_cols].fillna(0))
print('Scaling applied (StandardScaler fit on train only)')


In [ ]:
# ── Tokenise moves (only needed if a sequence model is selected) ────────────
need_seq = RUN_BILSTM or RUN_CNN or RUN_TRANSFORMER

if need_seq:
    MOVE_RE = re.compile(
        r'(?:\d+\.+\s*)?([KQRBN]?[a-h]?[1-8]?x?[a-h][1-8](?:=[QRBN])?[+#]?|O-O-O[+#]?|O-O[+#]?)'
    )
    move_to_id = {'<PAD>': 0, '<UNK>': 1}

    def tokenise(pgn):
        moves  = MOVE_RE.findall(pgn)
        clocks = extract_clocks(pgn)
        for m in moves:
            if m not in move_to_id:
                move_to_id[m] = len(move_to_id)
        return moves, clocks

    parsed           = df['Moves'].apply(tokenise)
    df['clean_moves']  = parsed.apply(lambda x: x[0])
    df['clean_clocks'] = parsed.apply(lambda x: x[1])
    df['n_moves']      = df['clean_moves'].apply(len)

    before = len(df)
    df     = df[df['n_moves'] >= MIN_PLIES]   # keep original index labels

    # Rebuild index sets after dropping short games
    keep    = set(df.index)
    idx_tr  = [i for i in idx_tr  if i in keep]
    idx_val = [i for i in idx_val if i in keep]
    idx_te  = [i for i in idx_te  if i in keep]

    def encode_move(m): return move_to_id.get(m, 1)
    def vocab_size():   return len(move_to_id)

    print(f'Dropped {before - len(df):,} games shorter than {MIN_PLIES} plies')
    print(f'Vocabulary: {vocab_size():,} distinct moves')
else:
    print('Tokenisation skipped (only Residual MLP selected)')
    def vocab_size(): return 2


In [ ]:
class ChessDataset(Dataset):
    # Holds pre-computed tensors for one data split.
    # with_seq=True also stores tokenised move sequences and normalised clocks.
    # The Residual MLP uses only 'static'; sequence models use all fields.

    def __init__(self, df_slice, static_cols, with_seq=True):
        self.with_seq = with_seq

        static_np = df_slice[static_cols].to_numpy(dtype=np.float32)
        w_dists   = np.stack(df_slice['WhiteElo'].apply(make_elo_dist).values)
        b_dists   = np.stack(df_slice['BlackElo'].apply(make_elo_dist).values)
        elos_np   = df_slice[['WhiteElo', 'BlackElo']].to_numpy(dtype=np.float32)

        self.static = torch.from_numpy(static_np)
        self.w_dist = torch.from_numpy(w_dists)
        self.b_dist = torch.from_numpy(b_dists)
        self.elos   = torch.from_numpy(elos_np)

        if with_seq:
            self.move_ids    = []
            self.norm_clocks = []
            for _, row in df_slice.iterrows():
                tc  = max(1, int(row.get('tc_base', 600)))
                ids = [encode_move(m) for m in row['clean_moves'][:MAX_SEQ_LEN]]
                n   = len(ids)
                raw = list(row['clean_clocks'])[:n]
                nrm = [min(max(c / tc, 0.), 2.) for c in raw] + [0.5] * (n - len(raw))
                self.move_ids.append(torch.tensor(ids, dtype=torch.long))
                self.norm_clocks.append(torch.tensor(nrm, dtype=torch.float32))

    def __len__(self):
        return len(self.static)

    def __getitem__(self, i):
        item = {'static': self.static[i], 'w_dist': self.w_dist[i],
                'b_dist': self.b_dist[i], 'elos': self.elos[i]}
        if self.with_seq:
            item['moves']  = self.move_ids[i]
            item['clocks'] = self.norm_clocks[i]
        return item


def collate(batch):
    # Pad variable-length sequences to the same length within each batch
    out = {
        'static': torch.stack([b['static'] for b in batch]),
        'w_dist': torch.stack([b['w_dist'] for b in batch]),
        'b_dist': torch.stack([b['b_dist'] for b in batch]),
        'elos':   torch.stack([b['elos']   for b in batch]),
    }
    if 'moves' in batch[0]:
        moves  = pad_sequence([b['moves']  for b in batch], batch_first=True, padding_value=0)
        clocks = pad_sequence([b['clocks'] for b in batch], batch_first=True, padding_value=0.5)
        out['moves']  = moves
        out['clocks'] = clocks
        out['mask']   = (moves != 0)   # True = real token
    return out


print('Building datasets...')
t0       = time.time()
train_ds = ChessDataset(df.loc[idx_tr],  static_cols, with_seq=need_seq)
val_ds   = ChessDataset(df.loc[idx_val], static_cols, with_seq=need_seq)
test_ds  = ChessDataset(df.loc[idx_te],  static_cols, with_seq=need_seq)
print(f'Built in {time.time()-t0:.0f}s')

# Weighted sampler: oversample rare Elo brackets
avg_tr  = (df.loc[idx_tr, 'WhiteElo'] + df.loc[idx_tr, 'BlackElo']) / 2
weights = cff.compute_elo_sample_weights(avg_tr)
sampler = WeightedRandomSampler(torch.FloatTensor(weights), len(train_ds))

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          collate_fn=collate, num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          collate_fn=collate, num_workers=0)
print(f'Train batches per epoch: {len(train_loader):,}')


## 4 · Model Architectures

### Development history

**v1–v2 — BiLSTM baseline (MAE ≈ 200)**  
Initial model: two-layer BiLSTM over move sequences plus hand-crafted static features
(captures, checks, castling, clock usage). Early experiments used a strict time-control
filter (10 min only) that left just ~23 k games — too few to generalise. Relaxing to ≥ 3 min
and scaling to 200 k → 1 M games brought MAE to ~200 Elo.

**v3 — Soft targets + KL divergence (MAE ≈ 185)**  
Switching from Huber regression to KL divergence over Gaussian soft targets improved
calibration and reduced MAE by ~15 Elo. Inspired by Ouzounis et al. (2022).

**v4 — Clock features + 1 M games (MAE ≈ 165)**  
Adding per-move clock statistics (time pressure, opening pace) and training on 1 M
games closed the gap further. A sequence autoencoder (BiLSTM → 32-d bottleneck) was
also tested to pre-compute sequence embeddings for faster CPU training.

**This notebook — Four architectures compared**  
All models use the same data, features, loss, and training loop for a fair comparison.


In [ ]:
def make_static_branch(n_in, dropout):
    # Small MLP: maps static feature vector to a 32-dim representation
    return nn.Sequential(
        nn.Linear(n_in, 128), nn.LayerNorm(128), nn.GELU(), nn.Dropout(dropout),
        nn.Linear(128, 64), nn.GELU(),
        nn.Linear(64,  32), nn.GELU(),
    )


### Model 1 · BiLSTM with soft attention

A two-layer **bidirectional LSTM** reads the move sequence and produces a hidden state
at each position. **Soft attention** learns to weight these states, allowing the model
to focus on the most informative moves.

- *Strength*: captures long-range move dependencies and is sensitive to move order.
- *Weakness*: sequential computation cannot be parallelised; can lose signal from
  early moves in long games.


In [ ]:
class BiLSTM(nn.Module):
    def __init__(self, n_vocab, embed_dim, hidden, n_static, dropout, n_bins):
        super().__init__()
        self.embed  = nn.Embedding(n_vocab, embed_dim, padding_idx=0)
        self.lstm   = nn.LSTM(embed_dim + 1, hidden, num_layers=2,
                              batch_first=True, bidirectional=True, dropout=dropout)
        self.attn   = nn.Linear(hidden * 2, 1)   # attention score per position
        self.static_net = make_static_branch(n_static, dropout)
        self.fusion = nn.Sequential(
            nn.Linear(hidden * 2 + 32, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout * 0.5),
        )
        self.head_w = nn.Linear(128, n_bins)
        self.head_b = nn.Linear(128, n_bins)
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                if m.bias is not None: nn.init.zeros_(m.bias)

    def forward(self, batch):
        e   = self.embed(batch['moves'])                           # (B, T, E)
        x   = torch.cat([e, batch['clocks'].unsqueeze(-1)], -1)   # (B, T, E+1)
        out, _ = self.lstm(x)                                     # (B, T, 2H)

        # Soft attention: score each position, mask padding, then softmax
        score = self.attn(out).squeeze(-1)                        # (B, T)
        score.masked_fill_(~batch['mask'], -1e9)
        alpha = score.softmax(1).unsqueeze(-1)                    # (B, T, 1)
        ctx   = (out * alpha).sum(1)                              # (B, 2H)

        h = self.fusion(torch.cat([ctx, self.static_net(batch['static'])], -1))
        return self.head_w(h), self.head_b(h)


### Model 2 · Residual MLP (features only)

A deep **residual MLP** over the static feature vector — no move sequences.

Each residual block applies `LayerNorm → Linear → GELU → Dropout → Linear`
and adds the result back via a skip connection. Pre-norm residual connections
train stably at depth and avoid gradient vanishing.

- *Strength*: fastest to train, no sequence overhead, very interpretable.
- *Weakness*: cannot see *which moves* were played, only aggregate statistics.


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, d, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.LayerNorm(d),
            nn.Linear(d, d * 2), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(d * 2, d),
        )
    def forward(self, x):
        return x + self.net(x)   # skip connection


class ResidualMLP(nn.Module):
    def __init__(self, n_static, hidden, n_blocks, dropout, n_bins):
        super().__init__()
        self.proj   = nn.Sequential(
            nn.Linear(n_static, hidden), nn.LayerNorm(hidden), nn.GELU(), nn.Dropout(dropout),
        )
        self.blocks = nn.Sequential(*[ResidualBlock(hidden, dropout) for _ in range(n_blocks)])
        self.head_w = nn.Linear(hidden, n_bins)
        self.head_b = nn.Linear(hidden, n_bins)

    def forward(self, batch):
        x = self.blocks(self.proj(batch['static']))
        return self.head_w(x), self.head_b(x)


### Model 3 · Multi-scale 1D CNN

**Three parallel Conv1d branches** with kernel sizes 3, 5, and 7 capture move
patterns at different scales: short tactics (k=3), medium plans (k=5), and longer
strategic ideas (k=7). After convolution, **global max-pooling** picks the strongest
filter activation across the whole game.

- *Strength*: fully parallelised, robust global max-pooling, best single-model MAE in our tests.
- *Weakness*: receptive field capped at 7 moves per layer; very long-range dependencies
  require deep stacking.

**Best single-model result: combined MAE ≈ 165 Elo.**


In [ ]:
class CNN(nn.Module):
    def __init__(self, n_vocab, embed_dim, n_filters, n_static, dropout, n_bins):
        super().__init__()
        self.embed    = nn.Embedding(n_vocab, embed_dim, padding_idx=0)
        in_ch         = embed_dim + 1   # move embedding + normalised clock
        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(in_ch, n_filters, kernel_size=k, padding=k // 2),
                nn.BatchNorm1d(n_filters),
                nn.GELU(),
            ) for k in (3, 5, 7)
        ])
        self.static_net = make_static_branch(n_static, dropout)
        self.fusion = nn.Sequential(
            nn.Linear(n_filters * 3 + 32, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout * 0.5),
        )
        self.head_w = nn.Linear(128, n_bins)
        self.head_b = nn.Linear(128, n_bins)

    def forward(self, batch):
        e    = self.embed(batch['moves'])                          # (B, T, E)
        x    = torch.cat([e, batch['clocks'].unsqueeze(-1)], -1)  # (B, T, E+1)
        x    = x.transpose(1, 2)                                  # (B, E+1, T) for Conv1d
        pad  = ~batch['mask']                                     # (B, T) True = padding

        parts = []
        for branch in self.branches:
            feat = branch(x)                                      # (B, F, T)
            feat.masked_fill_(pad.unsqueeze(1), float('-inf'))    # ignore padding in max-pool
            parts.append(feat.max(dim=-1).values)                 # (B, F)

        ctx = torch.cat(parts, dim=-1)                            # (B, 3F)
        h   = self.fusion(torch.cat([ctx, self.static_net(batch['static'])], -1))
        return self.head_w(h), self.head_b(h)


### Model 4 · Transformer Encoder

**Multi-head self-attention** lets every move attend to every other move in O(1),
unlike LSTM which passes information through a chain of sequential states.

We use **sinusoidal positional encodings** (fixed, not learned) and **pre-norm layers**
(LayerNorm before the attention/FFN blocks) which train more stably at depth.
After encoding, we **mean-pool** over non-padding positions.

- *Strength*: direct connections between any pair of moves; highly parallelisable on GPU.
- *Weakness*: quadratic memory in sequence length; advantage over LSTM is less clear
  on short sequences (most games are ≤ 60 moves).


In [ ]:
class Transformer(nn.Module):
    def __init__(self, n_vocab, embed_dim, d_model, n_heads, n_layers, ff_dim,
                 n_static, dropout, n_bins):
        super().__init__()
        self.embed      = nn.Embedding(n_vocab, embed_dim, padding_idx=0)
        self.input_proj = nn.Linear(embed_dim + 1, d_model)   # +1 for clock

        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=ff_dim,
            dropout=dropout, batch_first=True, norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.register_buffer('pos_enc', self._sinusoidal(512, d_model))

        self.static_net = make_static_branch(n_static, dropout)
        self.fusion = nn.Sequential(
            nn.Linear(d_model + 32, 256), nn.LayerNorm(256), nn.GELU(), nn.Dropout(dropout),
            nn.Linear(256, 128), nn.GELU(), nn.Dropout(dropout * 0.5),
        )
        self.head_w = nn.Linear(128, n_bins)
        self.head_b = nn.Linear(128, n_bins)

    @staticmethod
    def _sinusoidal(max_len, d_model):
        # Fixed positional encoding: sin/cos at different frequencies
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        enc = torch.zeros(max_len, d_model)
        enc[:, 0::2] = torch.sin(pos * div)
        enc[:, 1::2] = torch.cos(pos * div)
        return enc.unsqueeze(0)   # (1, max_len, d_model)

    def forward(self, batch):
        e   = self.embed(batch['moves'])                           # (B, T, E)
        x   = torch.cat([e, batch['clocks'].unsqueeze(-1)], -1)   # (B, T, E+1)
        x   = self.input_proj(x) + self.pos_enc[:, :x.size(1)]   # (B, T, d_model)

        key_pad = ~batch['mask']                                   # True = ignore
        x = self.encoder(x, src_key_padding_mask=key_pad)         # (B, T, d_model)

        # Mean-pool over real (non-padding) tokens
        lengths = batch['mask'].sum(1, keepdim=True).float()      # (B, 1)
        x.masked_fill_(key_pad.unsqueeze(-1), 0)
        ctx = x.sum(1) / lengths.clamp(min=1)                     # (B, d_model)

        h = self.fusion(torch.cat([ctx, self.static_net(batch['static'])], -1))
        return self.head_w(h), self.head_b(h)


## 5 · Training

All models share the same training loop:
- **Loss**: KL divergence between log-softmax predictions and the Gaussian soft targets
- **Optimiser**: AdamW with cosine annealing (LR decays to 5% of initial)
- **Gradient clipping** at norm 1.0
- **Early stopping**: restores best weights if validation MAE does not improve
  for `PATIENCE` consecutive epochs


In [ ]:
def get_mae(model, loader):
    # Quick MAE computation used during validation
    model.eval()
    bins = torch.tensor(BIN_CENTERS, device=device)
    errs = []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            w_out, b_out = model(batch)
            w_elo = (w_out.softmax(-1) * bins).sum(-1)
            b_elo = (b_out.softmax(-1) * bins).sum(-1)
            errs.append(((w_elo - batch['elos'][:, 0]).abs() +
                         (b_elo - batch['elos'][:, 1]).abs()) / 2)
    return torch.cat(errs).mean().item()


def train_model(model, name):
    opt   = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS, eta_min=LR * 0.05)
    kl    = nn.KLDivLoss(reduction='batchmean')

    best_mae, best_state, patience, history = float('inf'), None, 0, []

    print(f'\nTraining {name}')
    print('-' * 40)
    for epoch in range(NUM_EPOCHS):
        model.train()
        total_loss = 0
        for batch in train_loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            w_out, b_out = model(batch)
            loss = (kl(w_out.log_softmax(-1), batch['w_dist']) +
                    kl(b_out.log_softmax(-1), batch['b_dist']))
            opt.zero_grad()
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total_loss += loss.item()

        val_mae  = get_mae(model, val_loader)
        avg_loss = total_loss / len(train_loader)
        sched.step()
        history.append({'epoch': epoch + 1, 'loss': avg_loss, 'val_mae': val_mae})
        print(f'  {epoch+1:2d}/{NUM_EPOCHS}  loss={avg_loss:.4f}  val_mae={val_mae:.1f}')

        if val_mae < best_mae - 0.5:
            best_mae   = val_mae
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience   = 0
        else:
            patience += 1
            if patience >= PATIENCE:
                print(f'  Early stop at epoch {epoch+1}')
                break

    model.load_state_dict(best_state)
    return history


def eval_model(model, loader):
    # Full evaluation: collects predictions and distributions for analysis
    model.eval()
    bins = torch.tensor(BIN_CENTERS, device=device)
    w_preds, b_preds, w_true, b_true = [], [], [], []
    w_probs_all, b_probs_all = [], []
    with torch.no_grad():
        for batch in loader:
            batch  = {k: v.to(device) for k, v in batch.items()}
            w_out, b_out = model(batch)
            wp = w_out.softmax(-1);  bp = b_out.softmax(-1)
            w_preds.append((wp * bins).sum(-1).cpu())
            b_preds.append((bp * bins).sum(-1).cpu())
            w_true.append(batch['elos'][:, 0].cpu())
            b_true.append(batch['elos'][:, 1].cpu())
            w_probs_all.append(wp.cpu())
            b_probs_all.append(bp.cpu())

    wp = torch.cat(w_preds).numpy();   bp  = torch.cat(b_preds).numpy()
    wt = torch.cat(w_true).numpy();    bt  = torch.cat(b_true).numpy()
    mae_w = float(np.abs(wp - wt).mean())
    mae_b = float(np.abs(bp - bt).mean())
    return {
        'mae_white': mae_w, 'mae_black': mae_b, 'combined_mae': (mae_w + mae_b) / 2,
        'w_pred': wp, 'b_pred': bp, 'w_true': wt, 'b_true': bt,
        'w_probs': torch.cat(w_probs_all).numpy(),
        'b_probs': torch.cat(b_probs_all).numpy(),
    }


## 5a · Hyperparameter Optimisation (optional)

When `HPO_*` is enabled for a model, [Optuna](https://optuna.org/) runs a
quick search before the full training. Each trial trains on a small subset
(`HPO_GAMES`) for a few epochs (`HPO_EPOCHS`) and evaluates on the validation set.

The best parameters are cached to `checkpoints/hpo_*.pkl` — re-running the
cell is instant if the cache exists (delete the file to force a fresh search).

If HPO is disabled, the default values from the config cell are used.

**Searched parameters:**
- *BiLSTM*: hidden dim, dropout
- *Residual MLP*: hidden dim, number of residual blocks, dropout
- *CNN*: filters per branch, embedding dim, dropout
- *Transformer*: d\_model, attention heads, encoder layers, feedforward dim, dropout


In [ ]:
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ── Default params — overridden below when HPO is enabled ──────────────────
bilstm_p   = {'hidden':    BILSTM_HIDDEN,   'dropout': DROPOUT}
residual_p = {'hidden':    RESIDUAL_DIM, 'n_blocks': RESIDUAL_BLOCKS, 'dropout': DROPOUT}
cnn_p      = {'n_filters': CNN_FILTERS, 'embed_dim': EMBED_DIM, 'dropout': DROPOUT}
trans_p    = {'d_model':   TRANS_DIM, 'n_heads': TRANS_HEADS,
              'n_layers':  TRANS_LAYERS, 'ff_dim': TRANS_FF_DIM, 'dropout': DROPOUT}

need_hpo = ((RUN_BILSTM and HPO_BILSTM) or (RUN_RESIDUAL and HPO_RESIDUAL) or
            (RUN_CNN    and HPO_CNN)     or (RUN_TRANSFORMER and HPO_TRANSFORMER))

if need_hpo:
    # Small subset for fast trials
    rng        = np.random.default_rng(SEED)
    hpo_idx    = list(rng.choice(idx_tr, size=min(HPO_GAMES, len(idx_tr)), replace=False))
    hpo_ds     = ChessDataset(df.loc[hpo_idx], static_cols, with_seq=need_seq)
    hpo_loader = DataLoader(hpo_ds, batch_size=BATCH_SIZE, shuffle=True,
                            collate_fn=collate, num_workers=0)

    def quick_eval(model):
        # Short training on HPO subset, then MAE on the full validation set
        opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
        kl  = nn.KLDivLoss(reduction='batchmean')
        model.train()
        for _ in range(HPO_EPOCHS):
            for batch in hpo_loader:
                batch = {k: v.to(device) for k, v in batch.items()}
                w_out, b_out = model(batch)
                loss = (kl(w_out.log_softmax(-1), batch['w_dist']) +
                        kl(b_out.log_softmax(-1), batch['b_dist']))
                opt.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                opt.step()
        return get_mae(model, val_loader)

    # ── BiLSTM ─────────────────────────────────────────────────────────────
    if RUN_BILSTM and HPO_BILSTM:
        cache = CKPT_DIR / 'hpo_bilstm.pkl'
        if cache.exists():
            bilstm_p = pickle.loads(cache.read_bytes())
            print(f'BiLSTM HPO (cached): {bilstm_p}')
        else:
            print(f'BiLSTM HPO — {HPO_TRIALS} trials ...')
            def bilstm_obj(trial):
                p = {'hidden':  trial.suggest_categorical('hidden',  [64, 128, 256]),
                     'dropout': trial.suggest_float('dropout', 0.1, 0.4)}
                m = BiLSTM(vocab_size(), EMBED_DIM, p['hidden'],
                           n_static, p['dropout'], N_BINS).to(device)
                return quick_eval(m)
            study = optuna.create_study(direction='minimize')
            study.optimize(bilstm_obj, n_trials=HPO_TRIALS, show_progress_bar=True)
            bilstm_p = study.best_params
            cache.write_bytes(pickle.dumps(bilstm_p))
            print(f'BiLSTM best: {bilstm_p}')

    # ── Residual MLP ────────────────────────────────────────────────────────
    if RUN_RESIDUAL and HPO_RESIDUAL:
        cache = CKPT_DIR / 'hpo_residual.pkl'
        if cache.exists():
            residual_p = pickle.loads(cache.read_bytes())
            print(f'Residual HPO (cached): {residual_p}')
        else:
            print(f'Residual HPO — {HPO_TRIALS} trials ...')
            def residual_obj(trial):
                p = {'hidden':   trial.suggest_categorical('hidden',   [128, 256, 512]),
                     'n_blocks': trial.suggest_categorical('n_blocks', [2, 4, 6, 8]),
                     'dropout':  trial.suggest_float('dropout', 0.1, 0.4)}
                m = ResidualMLP(n_static, p['hidden'], p['n_blocks'],
                                p['dropout'], N_BINS).to(device)
                return quick_eval(m)
            study = optuna.create_study(direction='minimize')
            study.optimize(residual_obj, n_trials=HPO_TRIALS, show_progress_bar=True)
            residual_p = study.best_params
            cache.write_bytes(pickle.dumps(residual_p))
            print(f'Residual best: {residual_p}')

    # ── CNN ─────────────────────────────────────────────────────────────────
    if RUN_CNN and HPO_CNN:
        cache = CKPT_DIR / 'hpo_cnn.pkl'
        if cache.exists():
            cnn_p = pickle.loads(cache.read_bytes())
            print(f'CNN HPO (cached): {cnn_p}')
        else:
            print(f'CNN HPO — {HPO_TRIALS} trials ...')
            def cnn_obj(trial):
                p = {'n_filters': trial.suggest_categorical('n_filters', [64, 128, 256]),
                     'embed_dim': trial.suggest_categorical('embed_dim', [64, 128]),
                     'dropout':   trial.suggest_float('dropout', 0.1, 0.4)}
                m = CNN(vocab_size(), p['embed_dim'], p['n_filters'],
                        n_static, p['dropout'], N_BINS).to(device)
                return quick_eval(m)
            study = optuna.create_study(direction='minimize')
            study.optimize(cnn_obj, n_trials=HPO_TRIALS, show_progress_bar=True)
            cnn_p = study.best_params
            cache.write_bytes(pickle.dumps(cnn_p))
            print(f'CNN best: {cnn_p}')

    # ── Transformer ──────────────────────────────────────────────────────────
    if RUN_TRANSFORMER and HPO_TRANSFORMER:
        cache = CKPT_DIR / 'hpo_transformer.pkl'
        if cache.exists():
            trans_p = pickle.loads(cache.read_bytes())
            print(f'Transformer HPO (cached): {trans_p}')
        else:
            print(f'Transformer HPO — {HPO_TRIALS} trials ...')
            def trans_obj(trial):
                n_heads  = trial.suggest_categorical('n_heads',  [4, 8])
                d_model  = trial.suggest_categorical('d_model',  [64, 128, 256])
                d_model  = (d_model // n_heads) * n_heads   # must be divisible
                p = {'n_heads':  n_heads, 'd_model':  d_model,
                     'n_layers': trial.suggest_categorical('n_layers', [2, 4, 6]),
                     'ff_dim':   trial.suggest_categorical('ff_dim',   [128, 256, 512]),
                     'dropout':  trial.suggest_float('dropout', 0.1, 0.4)}
                m = Transformer(vocab_size(), EMBED_DIM, p['d_model'], p['n_heads'],
                                p['n_layers'], p['ff_dim'], n_static, p['dropout'],
                                N_BINS).to(device)
                return quick_eval(m)
            study = optuna.create_study(direction='minimize')
            study.optimize(trans_obj, n_trials=HPO_TRIALS, show_progress_bar=True)
            trans_p = study.best_params
            trans_p['d_model'] = (trans_p['d_model'] // trans_p['n_heads']) * trans_p['n_heads']
            cache.write_bytes(pickle.dumps(trans_p))
            print(f'Transformer best: {trans_p}')

else:
    print('HPO disabled — using default hyperparameters from config')

print(f'\nFinal params:')
if RUN_BILSTM:      print(f'  BiLSTM      : {bilstm_p}')
if RUN_RESIDUAL:    print(f'  Residual MLP: {residual_p}')
if RUN_CNN:         print(f'  CNN         : {cnn_p}')
if RUN_TRANSFORMER: print(f'  Transformer : {trans_p}')


In [ ]:
results   = {}   # model_name -> evaluation dict
histories = {}   # model_name -> training history list

def run(name, model):
    model = model.to(device)
    h     = train_model(model, name)
    r     = eval_model(model, test_loader)
    torch.save(model.state_dict(), CKPT_DIR / f'{name.replace(" ", "_")}.pth')
    results[name]   = r
    histories[name] = h
    print(f'  Test MAE: {r["combined_mae"]:.1f}')
    return model


if RUN_BILSTM:
    bilstm_model = run('BiLSTM',
        BiLSTM(vocab_size(), EMBED_DIM,
               bilstm_p['hidden'], n_static, bilstm_p['dropout'], N_BINS))

if RUN_RESIDUAL:
    residual_model = run('Residual MLP',
        ResidualMLP(n_static,
                    residual_p['hidden'], residual_p['n_blocks'], residual_p['dropout'], N_BINS))

if RUN_CNN:
    cnn_model = run('CNN',
        CNN(vocab_size(), cnn_p['embed_dim'], cnn_p['n_filters'],
            n_static, cnn_p['dropout'], N_BINS))

if RUN_TRANSFORMER:
    transformer_model = run('Transformer',
        Transformer(vocab_size(), EMBED_DIM,
                    trans_p['d_model'], trans_p['n_heads'], trans_p['n_layers'],
                    trans_p['ff_dim'], n_static, trans_p['dropout'], N_BINS))


## 6 · Results & Comparison


In [ ]:
def pct_within(r, thr):
    return 100 * np.mean(
        (np.abs(r['w_pred'] - r['w_true']) < thr) &
        (np.abs(r['b_pred'] - r['b_true']) < thr)
    )

rows = []
for name, r in results.items():
    rows.append({
        'Model':        name,
        'White MAE':   f"{r['mae_white']:.1f}",
        'Black MAE':   f"{r['mae_black']:.1f}",
        'Combined MAE': f"{r['combined_mae']:.1f}",
        'Within 100':  f"{pct_within(r, 100):.1f}%",
        'Within 200':  f"{pct_within(r, 200):.1f}%",
        'Within 300':  f"{pct_within(r, 300):.1f}%",
    })

summary = pd.DataFrame(rows).set_index('Model')
print(summary.to_string())


In [ ]:
if histories:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    colors = plt.cm.tab10.colors
    for i, (name, h) in enumerate(histories.items()):
        c = colors[i % len(colors)]
        axes[0].plot([e['epoch']   for e in h], [e['loss']    for e in h],
                     '-o', label=name, color=c, markersize=4)
        axes[1].plot([e['epoch']   for e in h], [e['val_mae'] for e in h],
                     '-o', label=name, color=c, markersize=4)
    axes[0].set_title('Training Loss (KL divergence)'); axes[0].set_xlabel('Epoch')
    axes[1].set_title('Validation MAE (Elo)');          axes[1].set_xlabel('Epoch')
    for ax in axes: ax.legend(fontsize=9)
    plt.tight_layout(); plt.show()


In [ ]:
if results:
    names = list(results.keys())
    maes  = [results[n]['combined_mae'] for n in names]
    fig, ax = plt.subplots(figsize=(7, 4))
    bars = ax.bar(names, maes, color=plt.cm.tab10.colors[:len(names)],
                  edgecolor='white', width=0.5)
    for bar, val in zip(bars, maes):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                f'{val:.1f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
    ax.set_ylabel('Combined MAE (Elo)')
    ax.set_title('Model Comparison — Test Set')
    ax.set_ylim(0, max(maes) * 1.15)
    plt.tight_layout(); plt.show()


In [ ]:
if results:
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    colors = plt.cm.tab10.colors
    for i, (name, r) in enumerate(results.items()):
        c = colors[i % len(colors)]
        for ax, pred, true, title in [
            (axes[0], r['w_pred'], r['w_true'], 'White'),
            (axes[1], r['b_pred'], r['b_true'], 'Black'),
        ]:
            errs = np.sort(np.abs(pred - true))
            ys   = np.linspace(0, 100, len(errs))
            ax.plot(errs, ys, label=name, color=c, lw=2)
    for ax, title in zip(axes, ['White', 'Black']):
        for thr, ls in [(100, '--'), (200, ':')]:
            ax.axvline(thr, color='grey', ls=ls, lw=1, alpha=0.7, label=f'{thr} Elo')
        ax.set_xlabel('Absolute error (Elo)'); ax.set_ylabel('% of games')
        ax.set_title(f'Error CDF — {title}'); ax.legend(fontsize=8)
    plt.tight_layout(); plt.show()


In [ ]:
if results:
    edges  = [0, 1000, 1200, 1400, 1600, 1800, 2000, 2200, 4001]
    labels = ['<1k','1k-1.2k','1.2-1.4k','1.4-1.6k','1.6-1.8k','1.8-2k','2-2.2k','>2.2k']
    fig, ax = plt.subplots(figsize=(12, 4))
    x = np.arange(len(labels))
    w = 0.8 / max(len(results), 1)
    colors = plt.cm.tab10.colors
    for i, (name, r) in enumerate(results.items()):
        bracket_maes = []
        for lo, hi in zip(edges[:-1], edges[1:]):
            mask = (r['w_true'] >= lo) & (r['w_true'] < hi)
            bracket_maes.append(
                np.abs(r['w_pred'][mask] - r['w_true'][mask]).mean() if mask.sum() > 10 else np.nan
            )
        offset = (i - len(results) / 2) * w + w / 2
        ax.bar(x + offset, bracket_maes, width=w * 0.9,
               label=name, color=colors[i % len(colors)], alpha=0.85)
    ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
    ax.set_ylabel('MAE (Elo)'); ax.set_title('MAE by Elo Bracket (White)')
    ax.legend(); plt.tight_layout(); plt.show()


## 7 · Ensemble

When multiple models are trained, we average their **predicted distributions**
(not just the point estimates). Because each model's errors are partially independent,
ensemble averaging typically improves both accuracy and calibration.


In [ ]:
if len(results) > 1:
    ref = next(iter(results.values()))
    w_avg = np.mean([r['w_probs'] for r in results.values()], axis=0)
    b_avg = np.mean([r['b_probs'] for r in results.values()], axis=0)

    w_ens = (w_avg * BIN_CENTERS).sum(axis=1)
    b_ens = (b_avg * BIN_CENTERS).sum(axis=1)
    mae_w = float(np.abs(w_ens - ref['w_true']).mean())
    mae_b = float(np.abs(b_ens - ref['b_true']).mean())
    combined = (mae_w + mae_b) / 2

    results['Ensemble'] = {
        'mae_white': mae_w, 'mae_black': mae_b, 'combined_mae': combined,
        'w_pred': w_ens, 'b_pred': b_ens,
        'w_true': ref['w_true'], 'b_true': ref['b_true'],
        'w_probs': w_avg, 'b_probs': b_avg,
    }
    print(f'Ensemble combined MAE: {combined:.1f}  (White {mae_w:.1f}, Black {mae_b:.1f})')
else:
    print('Need at least 2 models for an ensemble.')


In [ ]:
rows = []
for name, r in results.items():
    rows.append({
        'Model':        name,
        'Combined MAE': f"{r['combined_mae']:.1f}",
        'Within 100':  f"{pct_within(r, 100):.1f}%",
        'Within 200':  f"{pct_within(r, 200):.1f}%",
        'Within 300':  f"{pct_within(r, 300):.1f}%",
    })
final_df = pd.DataFrame(rows).set_index('Model')
best = min((n for n in results if n != 'Ensemble'), key=lambda n: results[n]['combined_mae'],
           default=None)
print('=' * 50)
print(final_df.to_string())
print('=' * 50)
if best:
    print(f'Best single model: {best}  (MAE = {results[best]["combined_mae"]:.1f})')


## 8 · Single Game Demo

Paste any PGN string below and run the cell to predict both players' Elo ratings
using every trained model.


In [ ]:
DEMO_PGN = (
    '1. e4 e5 2. Nf3 Nc6 3. Bc4 Bc5 4. c3 Nf6 5. d4 exd4 6. cxd4 Bb4+ '
    '7. Nc3 Nxe4 8. O-O Bxc3 9. d5 Bf6 10. Re1 Ne7 11. Rxe4 d6 '
    '12. Bg5 Bxg5 13. Nxg5 O-O 14. Nxh7 *'
)

def predict_game(pgn, model, model_name):
    # Extract features for a single game
    feat   = cff.extract_features(pgn, '*')
    clk    = clock_features(pgn, 600)
    row    = {**feat, **clk, 'tc_base': 600, 'tc_inc': 0}
    static = np.array([row.get(c, 0.0) for c in static_cols], dtype=np.float32)
    static = scaler.transform(static.reshape(1, -1))
    static_t = torch.tensor(static, dtype=torch.float32, device=device)

    batch = {'static': static_t,
             'w_dist': torch.zeros(1, N_BINS, device=device),
             'b_dist': torch.zeros(1, N_BINS, device=device),
             'elos':   torch.zeros(1, 2,      device=device)}

    if need_seq:
        moves, clocks_raw = tokenise(pgn)
        ids  = [encode_move(m) for m in moves[:MAX_SEQ_LEN]]
        n    = len(ids)
        nrm  = [min(max(c/600, 0.), 2.) for c in clocks_raw[:n]] + [0.5]*(n - len(clocks_raw[:n]))
        batch['moves']  = torch.tensor(ids, dtype=torch.long,    device=device).unsqueeze(0)
        batch['clocks'] = torch.tensor(nrm, dtype=torch.float32, device=device).unsqueeze(0)
        batch['mask']   = (batch['moves'] != 0)

    model.eval()
    bins = torch.tensor(BIN_CENTERS, device=device)
    with torch.no_grad():
        w_out, b_out = model(batch)
    w_elo = float((w_out.softmax(-1) * bins).sum())
    b_elo = float((b_out.softmax(-1) * bins).sum())
    return w_elo, b_elo


print(f'{'Model':<18}  {'White Elo':>10}  {'Black Elo':>10}')
print('-' * 44)
trained = {}
if RUN_BILSTM      and 'bilstm_model'      in dir(): trained['BiLSTM']      = bilstm_model
if RUN_RESIDUAL    and 'residual_model'    in dir(): trained['Residual MLP'] = residual_model
if RUN_CNN         and 'cnn_model'         in dir(): trained['CNN']          = cnn_model
if RUN_TRANSFORMER and 'transformer_model' in dir(): trained['Transformer']  = transformer_model
for name, model in trained.items():
    w, b = predict_game(DEMO_PGN, model, name)
    print(f'  {name:<16}  {w:>10.0f}  {b:>10.0f}')


In [ ]:
# # ...existing code...
# import subprocess, datetime, sys, os

# nb_path  = os.path.abspath('chess_elo_final.ipynb')
# out_dir  = os.path.abspath('output_files')
# os.makedirs(out_dir, exist_ok=True)
# ts       = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
# out_html = os.path.join(out_dir, f'chess_elo_final_{MAX_GAMES:.0e}_{ts}.html')

# try:
#     # Use sys.executable to ensure we use the jupyter installed in the current python env
#     subprocess.run([sys.executable, '-m', 'jupyter', 'nbconvert', '--to', 'html', '--no-input',
#                     '--output', out_html, nb_path], check=True, capture_output=True, text=True)
#     print(f'Exported: {out_html}')
# except subprocess.CalledProcessError as e:
#     print("Failed to export notebook. Jupyter Error:")
#     print(e.stderr)
# # ...existing code...